In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'cm'

# ── 1. Parse BAND.dat ──────────────────────────────────────────────────────
# BAND.dat 포맷: 헤더(#)와 빈 줄로 밴드 구분
# 컬럼: K-path  Spin-Up(eV)  Spin-Down(eV)
bands = []
current = []
with open('BAND.dat') as f:
    for line in f:
        line = line.strip()
        if line.startswith('#') or line == '':
            if current:
                bands.append(np.array(current))
                current = []
        else:
            vals = list(map(float, line.split()))
            current.append(vals)
    if current:
        bands.append(np.array(current))

print(f"Bands: {len(bands)},  K-points: {len(bands[0])}")

#("{/Symbol G}"0.000, "X|Y"0.998, "{/Symbol G}"1.948, "Z|R"2.562, "{/Symbol G}"4.071, "T|U"5.202, "{/Symbol G}"6.373, "V"7.752)

# ── 2. High-symmetry k-positions and labels ────────────────────────────────
hs_positions = [0.000, 1.131, 2.303]
hs_labels    = ['T', 'Γ', 'U']

#hs_positions = [0.000, 0.998,1.948, 2.562, 4.071, 5.202, 6.373, 7.752]
#hs_labels    = ['Γ', 'X|Y', 'Γ', 'Z|R', 'Γ', 'T|U', 'Γ', 'V']

# ── 3. Detect discontinuities (k-point repeated at segment boundaries) ─────
# VASP BAND.dat 에서 high-sym point에서 k값이 연속으로 두 번 나오며
# 이 사이를 연결하면 가짜 수직선이 생김 → 해당 segment를 skip
kk0 = bands[0][:, 0]
skip_segments = {i for i in range(len(kk0) - 1) if abs(kk0[i] - kk0[i+1]) < 1e-5}

# ── 4. Plot ────────────────────────────────────────────────────────────────
THRESHOLD = 0.03   # eV: spin splitting 이하면 검정으로 그림
YMIN, YMAX = -4, 4
LW = 1.5

fig, ax = plt.subplots(figsize=(4,6))

for band in bands:
    kk  = band[:, 0]
    eup = band[:, 1]
    edn = band[:, 2]

    for i in range(len(kk) - 1):
        if i in skip_segments:
            continue  # high-sym 경계의 가짜 수직선 방지

        # 인접 두 k-point의 평균 spin splitting으로 색 결정
        diff_avg = 0.5 * (abs(eup[i] - edn[i]) + abs(eup[i+1] - edn[i+1]))
        if diff_avg < THRESHOLD:
            col_up = col_dn = 'black'
        else:
            col_up = '#FF0000'   # spin up: red
            col_dn = '#0000FF'   # spin down: blue

        seg_x = [kk[i], kk[i+1]]
        ax.plot(seg_x, [eup[i], eup[i+1]], color=col_up, lw=LW, solid_capstyle='round')
        ax.plot(seg_x, [edn[i], edn[i+1]], color=col_dn, lw=LW, solid_capstyle='round')

# ── 5. Axes & decorations ──────────────────────────────────────────────────
ax.set_ylim(YMIN, YMAX)
ax.set_xlim(hs_positions[0], hs_positions[-1])
ax.margins(x=0)

# Dashed vertical lines at high-symmetry points
#for xp in hs_positions:
ax.axvline(x=hs_positions[1], color='black', lw=1.0, linestyle='--', dashes=(4, 3), zorder=1)

# Fermi level
ax.axhline(y=0, color='black', ls='--', lw=1.2, zorder=1)

# X-axis labels (unicode Γ → upright, no mathtext italic)
ax.set_xticks(hs_positions)
ax.set_xticklabels(hs_labels, fontsize=20)
ax.tick_params(axis='x', which='both', length=0)

# Y-axis labels (explicit integer text)
ytick_vals = list(range(YMIN, YMAX + 1))
ax.set_yticks(ytick_vals)
ax.set_yticklabels([str(v) for v in ytick_vals], fontsize=20)
ax.tick_params(axis='y', length=4)

ax.set_ylabel(r'$\mathrm{E - E_F\ (eV)}$', fontsize=20)

plt.tight_layout()
plt.savefig('FeSi_bandstructure.png', dpi=200, bbox_inches='tight')
plt.savefig('FeSi_bandstructure.pdf', bbox_inches='tight')
print("Saved: bandstructure.png / bandstructure.pdf")